## User story
Archive older StoryMaps offline (to reduce online storage/credits) and optionally import them into a different org/portal later.

## What this notebook does
1. Signs in to a **source** portal/org (where the stories currently live).
2. Exports one or more StoryMap items to local `.contentexport` packages.
3. Signs in to a **target** portal/org (where you want to restore/migrate them).
4. Imports those packages into the target and prints the new item URLs.
5. Optionally cleans up local export files.

## Before you run it
- Create a `.env` file (or set env vars) with: `USERNAME_SOURCE`, `PASSWORD_SOURCE`, `USERNAME_TARGET`, `PASSWORD_TARGET`.
- Decide whether source/target are ArcGIS Online or Enterprise portals (set URLs below).
- Make sure you have permission to export (and to import/create content in the target).

In [ ]:
import arcgis
from arcgis.gis import GIS, Item
from dotenv import load_dotenv
from pathlib import Path
import os
import shutil

In [ ]:
EXPECTED_ARCGIS_VERSION = "2.4.3"
print("ArcGIS API for Python version:", arcgis.__version__)
if arcgis.__version__ != EXPECTED_ARCGIS_VERSION:
    print(
        "Note: this notebook was authored against",
        EXPECTED_ARCGIS_VERSION,
        "(you have",
        arcgis.__version__ + ")",
    )

In [ ]:
# Load credentials from .env (or OS environment variables)
load_dotenv()

USERNAME_SOURCE = os.getenv("USERNAME_SOURCE")
PASSWORD_SOURCE = os.getenv("PASSWORD_SOURCE")
USERNAME_TARGET = os.getenv("USERNAME_TARGET")
PASSWORD_TARGET = os.getenv("PASSWORD_TARGET")

missing = [
    name
    for name, value in {
        "USERNAME_SOURCE": USERNAME_SOURCE,
        "PASSWORD_SOURCE": PASSWORD_SOURCE,
        "USERNAME_TARGET": USERNAME_TARGET,
        "PASSWORD_TARGET": PASSWORD_TARGET,
    }.items()
    if not value
 ]
if missing:
    raise ValueError(f"Missing credentials: {', '.join(missing)}")

In [ ]:
# Portal URLs (ArcGIS Online shown here). For Enterprise, set to something like "https://portal.domain.com/portal"
SOURCE_PORTAL_URL = "https://www.arcgis.com"
TARGET_PORTAL_URL = "https://www.arcgis.com"

# StoryMap item IDs to export/import
storymap_item_ids = [""]  # Add StoryMap item IDs you want to export/import here

# Local folder where .contentexport packages will be written
export_folder = Path("exported_folder")

In [ ]:
# Create connections
gis_source = GIS(SOURCE_PORTAL_URL, USERNAME_SOURCE, PASSWORD_SOURCE)
gis_target = GIS(TARGET_PORTAL_URL, USERNAME_TARGET, PASSWORD_TARGET)

print("Signed in (source):", gis_source.properties.get("urlKey", ""), gis_source.users.me.username)
print("Signed in (target):", gis_target.properties.get("urlKey", ""), gis_target.users.me.username)

## Offline manager
ArcGIS provides an **offline manager** on a GIS connection that can:
- **Export** items to a local `.contentexport` package
- **Import** that package into a (possibly different) portal/org

> Note: Offline manager instances are tied to the GIS connection you create them from (source vs target).

In [ ]:
# Offline managers (bound to their respective GIS connections)
source_offline_manager = gis_source.content.offline
target_offline_manager = gis_target.content.offline

## Export StoryMaps (source → local packages)
This step exports each StoryMap item into `export_folder` as a `.contentexport` package.

You can archive these packages offline, or import them into another portal/org in the next step.

In [ ]:
# Export from source portal/org
export_folder.mkdir(parents=True, exist_ok=True)

items_to_export: list[Item] = [Item(gis_source, item_id) for item_id in storymap_item_ids]

print(f"Exporting {len(items_to_export)} item(s) to: {export_folder.resolve()}")
for item in items_to_export:
    print(f"- Exporting: {item.title} (ID: {item.id})")
    try:
        # Creates something like: exported_folder/item_<id>.contentexport
        source_offline_manager.export_items([item], str(export_folder), f"item_{item.id}")
        print(f"  ✓ Exported: {item.id}")
    except Exception as e:
        print(f"  ✗ Export failed for {item.id}: {e}")

## Import packages (local → target portal/org)
This step scans `export_folder` for `.contentexport` packages and imports them into the **target** portal/org.

It prints the created item titles/IDs/URLs so you can quickly verify the migration worked.

In [ ]:
# Import into target portal/org
if not export_folder.exists():
    raise FileNotFoundError(f"Export folder not found: {export_folder}")

packages = sorted(export_folder.glob("*.contentexport"))
if not packages:
    raise FileNotFoundError(f"No .contentexport packages found in: {export_folder}")

print(f"Importing {len(packages)} package(s) into target: {TARGET_PORTAL_URL}")
for package_path in packages:
    print(f"- Importing package: {package_path.name}")
    try:
        imported_items = target_offline_manager.import_content(package_path=str(package_path))
        for imported in imported_items:
            print(f"  ✓ Imported: {imported.title} (ID: {imported.id})")
            if getattr(imported, "url", None):
                print(f"    URL: {imported.url}")
    except Exception as e:
        print(f"  ✗ Import failed for {package_path.name}: {e}")

## Cleanup (optional)
Delete the local `export_folder` after you’ve verified the import worked.

In [ ]:
# Optional: remove local exports after validating results
# Uncomment to enable cleanup
# shutil.rmtree(export_folder)